## Import

In [2]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:0'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data generation

In [11]:
from datasets import NonlinearGaussian, MoG

n, d = 10000, 64                                 # < higher d, higher MI
true_rho = 0.7                                   # < higher rho, higher MI
case = '3b'                                      # < choose between ['1a', '1b', '2', '3a', '3b', '3c', 'MoG']

if case != 'MoG':
    dataset = NonlinearGaussian.NonlinearGaussian(n_samples=n, n_dims=d, rho=true_rho, mu=0, case=case)
    X0, Y0 = dataset.sample_data(n_samples = n)
    X, Y = dataset.transformation(X0, Y0)
    MI = dataset.true_mutual_info()              # we know GT MI
else:
    dataset = MoG.MoG(n_samples=n, n_dims=d, K=5, shifts=[-0.4, -0.1, 0, 0.1, 0.4], rhos=[0.5, 0.6, 0.7, 0.8, 0.9])
    X, Y = dataset.sample_data(n_samples = n)
    MI = dataset.empirical_mutual_info()         # MI by MC estimate

X, Y = X.to(device), Y.to(device)
Z = torch.cat([X, Y], dim=1)
T = torch.ones(n, 2).to(device)

print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", MI)

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 10.77351285222025


## MI estimation

In [17]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        self.max_iteration = 1250
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]

In [13]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(None, None, None, hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 1.0119751691818237 loss val= 0.9925312399864197 best val loss= 0.9925312399864197 best t= 0
finished: t= 63 loss= 0.6216835975646973 loss val= 0.6821123361587524 best val loss= 0.628451943397522 best t= 58
finished: t= 126 loss= 0.6096011996269226 loss val= 0.6083230972290039 best val loss= 0.6083230972290039 best t= 126
finished: t= 189 loss= 0.7769500613212585 loss val= 0.6921004056930542 best val loss= 0.5872188806533813 best t= 185
finished: t= 252 loss= 0.601024329662323 loss val= 0.6101611852645874 best val loss= 0.5781402587890625 best t= 248
finished: t= 315 loss= 0.6052427291870117 loss val= 0.613394558429718 best val loss= 0.5723422169685364 best t= 283
finished: t= 378 loss= 0.7309673428535461 loss val= 0.6012215614318848 best val loss= 0.5501247644424438 best t= 358
finished: t= 441 loss= 0.6511110663414001 loss val= 0.5855676531791687 best val loss= 0.5501247644424438 best t= 358
finished: t= 504 loss= 0.615402102470398 loss val= 0.5984602570533752 bes

In [14]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)


print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 1.8433828353881836 loss val= 2.306232452392578 best val loss= 2.306232452392578 best t= 0
finished: t= 63 loss= -0.6504611968994141 loss val= -0.7662544846534729 best val loss= -0.7662544846534729 best t= 63
finished: t= 126 loss= -1.114401936531067 loss val= -1.0291376113891602 best val loss= -1.0533335208892822 best t= 122
finished: t= 189 loss= -1.1964638233184814 loss val= -1.0473668575286865 best val loss= -1.2161129713058472 best t= 181
finished: t= 252 loss= -1.6823029518127441 loss val= -1.4644927978515625 best val loss= -1.4644927978515625 best t= 252
finished: t= 315 loss= -1.5411584377288818 loss val= -1.4806222915649414 best val loss= -1.5599313974380493 best t= 306
finished: t= 378 loss= -1.8232122659683228 loss val= -1.6569242477416992 best val loss= -1.7665724754333496 best t= 355
finished: t= 441 loss= -1.9070706367492676 loss val= -1.8024762868881226 best val loss= -1.9349443912506104 best t= 438
finished: t= 504 loss= -2.5317091941833496 loss val=

In [15]:
## Neural adaptive MI estimate
from estimators.VCE import VCE

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

K components= 5 copula transform= True
nde type: FM
finished: t= 0 loss= 219.8401336669922 loss val= 241.38650512695312 best val loss= 241.38650512695312 best t= 0
finished: t= 126 loss= 10.132242202758789 loss val= 13.901264190673828 best val loss= 12.2427978515625 best t= 124
finished: t= 252 loss= 12.700586318969727 loss val= 13.914348602294922 best val loss= 10.996676445007324 best t= 221
finished: t= 378 loss= 8.195488929748535 loss val= 11.425384521484375 best val loss= 9.82383918762207 best t= 326


finished: t= 0 loss= 996.8551635742188 loss val= 1059.4798583984375 best val loss= 1059.4798583984375 best t= 0
finished: t= 126 loss= 6.729680061340332 loss val= 8.388492584228516 best val loss= 6.688536643981934 best t= 124
finished: t= 252 loss= 10.94814682006836 loss val= 8.08071517944336 best val loss= 5.790534973144531 best t= 245
finished: t= 378 loss= 7.947507381439209 loss val= 7.319725036621094 best val loss= 5.664806842803955 best t= 341
finished: t= 504 loss= 5.7339301109